In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
pip install open_clip_torch pandas pillow annoy tqdm

In [ ]:
import os
import json
import torch
import open_clip
from PIL import Image
from annoy import AnnoyIndex
import numpy as np
from tqdm import tqdm

# 1. RUTAS LOCALES (Cambia estas rutas por las de tu ordenador)
# Ejemplo en Windows: r'C:\Usuarios\Sara\Fotos\Nuevas'
# Ejemplo en Mac/Linux: '/Users/Sara/Fotos/Nuevas'
PATH_CARPETA_LOCAL = '/content/drive/MyDrive/TFM-Sara/input/'
PATH_ANN = '/content/drive/MyDrive/TFM-Sara/support_data/index/support_database.ann'
PATH_JSON_METADATA = '/content/drive/MyDrive/TFM-Sara/support_data/index/metadata_index.json'
PATH_OUTPUT_GRAFO = '/content/drive/MyDrive/TFM-Sara/output/predicciones_año/grafo_predicciones.json'

# 2. CARGAR EL MODELO Y EL ÍNDICE
dimension = 512
annoy_index = AnnoyIndex(dimension, 'angular')
annoy_index.load(PATH_ANN)

with open(PATH_JSON_METADATA, 'r', encoding='utf-8') as f:
    support_metadata = json.load(f)

# Cargamos el modelo en CPU (si no tienes una gráfica potente en el PC)
device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model = model.to(device)
model.eval()

# 3. BUCLE DE PROCESAMIENTO — batch inference para máximo uso del GPU
# BATCH_SIZE=32 es óptimo para T4 (16 GB). Reducir si hay OOM.
BATCH_SIZE = 32
extensiones = ('.jpg', '.jpeg', '.png')
nodos_nuevos = {}

archivos_validos = sorted([
    a for a in os.listdir(PATH_CARPETA_LOCAL)
    if a.lower().endswith(extensiones)
])
print(f"Fotos a procesar: {len(archivos_validos)}  |  Batch size: {BATCH_SIZE}")

for batch_start in tqdm(range(0, len(archivos_validos), BATCH_SIZE),
                        desc="Batches OpenCLIP"):
    batch_archivos = archivos_validos[batch_start:batch_start + BATCH_SIZE]
    batch_tensors, batch_names = [], []

    for archivo in batch_archivos:
        ruta_completa = os.path.join(PATH_CARPETA_LOCAL, archivo)
        try:
            img = Image.open(ruta_completa).convert('RGB')
            batch_tensors.append(preprocess(img))
            batch_names.append(archivo)
        except Exception as e:
            print(f"Error leyendo {archivo}: {e}")

    if not batch_tensors:
        continue

    # Inferencia en bloque — 1 llamada GPU por batch en lugar de N
    batch_input = torch.stack(batch_tensors).to(device)
    with torch.no_grad():
        features_batch = model.encode_image(batch_input)
        features_batch /= features_batch.norm(dim=-1, keepdim=True)
    vectors = features_batch.cpu().numpy()

    for archivo, vector in zip(batch_names, vectors):
        try:
            ids_cercanos = annoy_index.get_nns_by_vector(vector, 5)
            datos_vecinos = [support_metadata[str(idx)] for idx in ids_cercanos]
            años = [v['year'] for v in datos_vecinos]

            nodos_nuevos[archivo] = {
                "id": archivo,
                "label": archivo,
                "attributes": {
                    "year_predicho": int(np.mean(años)),
                    "confianza": round(float(max(0.0, 100.0 - np.std(años))), 2)
                },
                "conexiones": [
                    {
                        "target": v['filename'],
                        "relacion": "similar_a",
                        "peso": 1.0 / (i + 1)
                    } for i, v in enumerate(datos_vecinos)
                ]
            }
        except Exception as e:
            print(f"Error procesando {archivo}: {e}")

# 4. EXPORTAR JSON PARA LA FUSIÓN
with open(PATH_OUTPUT_GRAFO, 'w', encoding='utf-8') as f:
    json.dump(nodos_nuevos, f, ensure_ascii=False, indent=4)

print(f"Proceso completado. Se han generado predicciones para {len(nodos_nuevos)} fotos.")

Procesando fotos locales desde: /content/


100%|██████████| 3/3 [00:00<00:00, 13965.50it/s]

Proceso completado. Se han generado predicciones para 0 fotos.


In [5]:
import networkx as nx
import json

# 1. Cargar tus predicciones
with open('/content/drive/MyDrive/TFM-Sara/output/predicciones_año/grafo_predicciones.json', 'r', encoding='utf-8') as f:
    predicciones = json.load(f)

# 2. Crear el objeto Grafo y añadir nodos
G = nx.Graph()

years_to_photos = {}

for foto_id_filename, info in predicciones.items(): # Rename to avoid confusion with internal ID
    # Create a unique internal node ID (e.g., a simple integer)
    internal_node_id = info['label']

    # Crear el nodo de la foto nueva con su nombre y año predicho
    año = info['attributes']['year_predicho']
    G.add_node(internal_node_id,
                label=foto_id_filename,
                year=año,
                confianza=info['attributes']['confianza'],
                group=1,
                size=25,
                type='circle',
                color='#4A90D9',
                dimension='imagen')

    if año not in years_to_photos:
        years_to_photos[año] = []
    years_to_photos[año].append(internal_node_id) # Store internal ID for connections

# 2.5. Add Year Nodes and connect them to photo nodes
for year, photos_in_year in years_to_photos.items():
    year_node_id = f"year_node_{year}"
    # Tamaño del hub proporcional al nº de fotos de ese año (mín 20, máx 60)
    hub_size = min(60, max(20, 20 + len(photos_in_year) * 2))
    G.add_node(year_node_id,
               label=str(year),
               group=3,
               type='square',
               color='#1A3A5C',
               size=hub_size,
               n_fotos=len(photos_in_year),
               dimension='año')

    for photo_node_id in photos_in_year:
        # Connect each photo to its corresponding year node with a strong weight
        G.add_edge(photo_node_id, year_node_id, relation='pertenece_a_año', weight=1.5, dimension='año')

# 3. Conexiones entre fotos del mismo año (constelaciones por época)
# Limitado a MAX_K vecinos por foto para evitar un grafo completo O(n²)
import random
random.seed(42)  # Semilla fija → resultados reproducibles entre ejecuciones
MAX_K = 5  # Cada foto conecta como máximo con 5 fotos de su mismo año

for year, photos_in_year in years_to_photos.items():
    if len(photos_in_year) < 2:
        continue
    for foto in photos_in_year:
        candidates = [p for p in photos_in_year if p != foto]
        neighbors = random.sample(candidates, min(MAX_K, len(candidates)))
        for neighbor in neighbors:
            if not G.has_edge(foto, neighbor):
                G.add_edge(foto, neighbor, relation='mismo_año', weight=0.8, dimension='año')

# 4. Guardar para visualizar
os.makedirs("/content/drive/MyDrive/TFM-Sara/output/predicciones_año", exist_ok=True)
nx.write_gexf(G, "/content/drive/MyDrive/TFM-Sara/output/predicciones_año/grafo_años.gexf")
n_años = sum(1 for _, d in G.nodes(data=True) if d.get("dimension") == "año")
n_imgs = sum(1 for _, d in G.nodes(data=True) if d.get("dimension") == "imagen")
print(f"✅ Grafo año: {n_imgs} imágenes · {n_años} hubs de año · {G.number_of_edges()} aristas")

✅ Grafo con conexiones intra-año y entre años cercanos generado. Ábrelo en Gephi para verlo visualmente.
